<!--
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Custom Kernels with Pallas

Pallas allows you to write custom TPU kernels in Python. This is essential for implementing new research ideas or where you need more fine-grained control over the code executed on the TPU hardware.

## 1. What is Pallas?

[Pallas](https://docs.jax.dev/en/latest/pallas/index.html) is a JAX-based kernel language for TPUs. It provides direct control over the TPU's memory hierarchy and vector units while remaining in pure Python.

`torch_tpu` integrates with Pallas via the `@pallas.jax_op` decorator, allowing you to write custom hardware-accelerated logic and call it directly within your PyTorch models.

**Prerequisites:** JAX must be installed (`pip install jax`).

In [1]:
import torch
import jax
from jax.experimental import pallas as pl
from torch_tpu._internal import pallas  # This will likely change from _internal to experimental

device = torch.device("tpu")

print(f"Pallas environment ready on {device}.")

I0000 00:00:1787340677.034388   78130 pjrt_api.cc:119] GetPjrtApi was found for TPU at /home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/libtpu/libtpu.so
I0000 00:00:1787340677.050912   78130 tpu_profiler_plugin.cc:791] PyTorch PrivateUse1ProfilerRegistry not found. Native TPU profiling will be disabled (this is expected on PyTorch < 2.12).
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
I0000 00:00:1787340677.072252   78130 device_rt.cc:130] PjRt runtime initialization deferred for tpu


Device type: tpu, Device index: default
Initializing TPU distributed runtime


Pallas environment ready on tpu.


## 2. Writing Your First Kernel

The `pallas.jax_op` function is the primary way to define a custom TPU kernel. By using it directly, you can bind your kernel logic to statically known output shapes.

### Key Concepts

This notebook is only focused on showing how to use Pallas in the context of TorchTPU. It is not meant to be a full tutorial on Pallas. If you are new to Pallas be sure to read the For more technical details on Pallas memory management, see the [Pallas Quickstart](https://docs.jax.dev/en/latest/pallas/quickstart.html).

- **`_ref` arguments**: In these examples, Pallas manages the transfer of data from HBM to local VMEM. These parameters act as references to that local memory.
- **`[...]` syntax**: Since Pallas handles the tile transfers here, we use `[...]` to load the full contents of the local reference into registers for computation, or to store them back.
- **`output_shapes`**: You provide example tensors (or shapes) to tell the PyTorch dispatcher how to allocate the output buffer statically.

In [2]:
# Define a simple vector addition kernel
def add_vectors_kernel(x_ref, y_ref, o_ref):
  """x_ref, y_ref: Input references

  o_ref: Output reference to be populated
  """
  x, y = x_ref[...], y_ref[...]
  o_ref[...] = x + y

x = torch.tensor([1.0, 2.0, 3.0], device="tpu")
y = torch.tensor([4.0, 5.0, 6.0], device="tpu")

# NEW WAY: Wrap the Pallas kernel in a JAX function
def add_vectors_jax(x: jax.Array, y: jax.Array) -> jax.Array:
  return pl.pallas_call(
      add_vectors_kernel, out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype)
  )(x, y)

# Wrap it using the functional interface
add_vectors_fn = pallas.jax_op(
    "pallas::add_vectors",
    add_vectors_jax,
)

# Use it like a standard PyTorch function
z = add_vectors_fn(x, y)

print(f"Input x:  {x.cpu()}")
print(f"Input y:  {y.cpu()}")
print(f"Result z: {z.cpu()}")

I0000 00:00:1787340677.386222   78130 pjrt_api.cc:168] The PJRT plugin has PJRT API version 0.114. The framework PJRT API version is 0.114.


I0000 00:00:1787340678.698938   78130 pjrt_c_api_client.cc:188] PjRtCApiClient created.


I0000 00:00:1787340678.969139   78457 compilation_cache_utils.cc:217] Tier-3 compilation cache root directory: 
I0000 00:00:1787340678.969158   78457 tier2_compilation_cache.cc:82] Tier-2 compilation cache is disabled for world size 1.
I0000 00:00:1787340678.969161   78457 tier3_compilation_cache.cc:50] Backup compilation for tier-3 cache read is disabled as tier-3 cache is disabled.


Input x:  tensor([1., 2., 3.])
Input y:  tensor([4., 5., 6.])


Result z: tensor([5., 7., 9.])


### 2.1 Multiple Outputs

If your kernel produces multiple results, simply pass a list of multiple example tensors to the `output_shapes` argument.

In [3]:
def add_subtract_kernel(x_ref, y_ref, out_add_ref, out_sub_ref):
  x, y = x_ref[...], y_ref[...]
  out_add_ref[...] = x + y
  out_sub_ref[...] = x - y

a = torch.tensor([10.0, 20.0, 30.0], device="tpu")
b = torch.tensor([1.0, 2.0, 3.0], device="tpu")

def add_subtract_jax(
    x: jax.Array, y: jax.Array
) -> tuple[jax.Array, jax.Array]:
  return pl.pallas_call(
      add_subtract_kernel,
      out_shape=[
          jax.ShapeDtypeStruct(x.shape, x.dtype),
          jax.ShapeDtypeStruct(x.shape, x.dtype),
      ],
  )(x, y)

add_subtract_fn = pallas.jax_op(
    "pallas::add_subtract",
    add_subtract_jax,
)

res_add, res_sub = add_subtract_fn(a, b)

print(f"Input a:    {a.cpu()}")
print(f"Input b:    {b.cpu()}")
print(f"a + b:      {res_add.cpu()}")
print(f"a - b:      {res_sub.cpu()}")

Input a:    tensor([10., 20., 30.])
Input b:    tensor([1., 2., 3.])
a + b:      tensor([11., 22., 33.])
a - b:      tensor([ 9., 18., 27.])


## 3. Integration: `torch.compile`

Pallas kernels can be seamlessly compiled into larger graphs using `torch.compile(backend="tpu")`. Because `pallas.jax_op` automatically registers the function as a `torch.library.custom_op` and handles the `register_fake` implementation internally, you can compile the resulting op directly.

In [4]:
from torch_tpu._internal import compile

# 1. Define the Pallas kernel and its JAX wrapper
def mul_kernel_fn(x_ref, y_ref, o_ref):
  o_ref[...] = x_ref[...] * y_ref[...]

def mul_jax_fn(x: jax.Array, y: jax.Array) -> jax.Array:
  return pl.pallas_call(
      mul_kernel_fn, out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype)
  )(x, y)

mul_vectors_op = pallas.jax_op("pallas::mul_vectors", mul_jax_fn)

# 4. Compile with the TPU backend
@torch.compile(backend="tpu")
def compiled_mul(x, y):
  return mul_vectors_op(x, y)

# --- Run it ---
x_inf = torch.tensor([1.0, 2.0, 3.0], device="tpu")
y_inf = torch.tensor([4.0, 5.0, 6.0], device="tpu")

res_compiled = compiled_mul(x_inf, y_inf)

print(f"Input x:  {x_inf.cpu()}")
print(f"Input y:  {y_inf.cpu()}")
print(f"Result:   {res_compiled.cpu()}")

Input x:  tensor([1., 2., 3.])
Input y:  tensor([4., 5., 6.])
Result:   tensor([ 4., 10., 18.])


/home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/torch/_inductor/codecache.py:1074: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  yield pickle.loads(content), content, True


## 4. Integration: Autograd

If you are using custom kernels in a training loop, you must manually define the backward pass and register it with the forward op using `register_autograd`.

Both the forward and backward passes should be defined using `pallas.jax_op`. This ensures they are registered as custom ops with their own `register_fake` implementations, which is required because AOTAutograd traces both the forward and backward graphs.

In [5]:
# 1. Define the backward Pallas kernel and its JAX wrapper
def mul_backward_kernel_fn(g_ref, x_ref, y_ref, g_x_ref, g_y_ref):
  g, x, y = g_ref[...], x_ref[...], y_ref[...]
  g_x_ref[...] = g * y
  g_y_ref[...] = g * x

def mul_backward_jax_fn(
    grad: jax.Array, x: jax.Array, y: jax.Array
) -> tuple[jax.Array, jax.Array]:
  return pl.pallas_call(
      mul_backward_kernel_fn,
      out_shape=[
          jax.ShapeDtypeStruct(x.shape, x.dtype),
          jax.ShapeDtypeStruct(y.shape, y.dtype),
      ],
  )(grad, x, y)

mul_vectors_backward_op = pallas.jax_op(
    "pallas::mul_vectors_backward", mul_backward_jax_fn
)

# 3. Register fake for the backward op
mul_vectors_backward_op.register_fake(
    lambda g, x, y: (torch.empty_like(x), torch.empty_like(y))
)

# 4. Connect forward and backward via autograd
def setup_context(ctx, inputs, output):
  x, y = inputs
  ctx.save_for_backward(x, y)

def backward_impl(ctx, grad):
  x, y = ctx.saved_tensors
  return mul_vectors_backward_op(grad, x, y)

mul_vectors_op.register_autograd(backward_impl, setup_context=setup_context)

# --- Run Training Loop ---
@torch.compile(dynamic=False, backend="tpu")
def compiled_mul_train(x, y):
  return mul_vectors_op(x, y).sum()

x_train = torch.tensor([1.0, 2.0, 3.0], device=device, requires_grad=True)
y_train = torch.tensor([4.0, 5.0, 6.0], device=device, requires_grad=True)

loss_compiled = compiled_mul_train(x_train, y_train)
loss_compiled.backward()

print(f"Loss:    {loss_compiled.detach().cpu().item()}")
print(f"x grad:  {x_train.grad.cpu()}")
print(f"y grad:  {y_train.grad.cpu()}")

/home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/torch_tpu/_internal/export/export.py:533: UserWarning: 'default_generators' is experimental and subject to change: Default generators for TPU devices.
  ] + [device_module.default_generators[device.index or 0]]


class GraphModule(torch.nn.Module):
    def forward(self, primals_1: "f32[3]", primals_2: "f32[3]"):
        # No stacktrace found for following nodes
        submod_0 = self.submod_0(primals_1, primals_2);  primals_1 = primals_2 = None
        getitem = submod_0[0]
        getitem_1 = submod_0[1]
        getitem_2 = submod_0[2];  submod_0 = None
        return (getitem, getitem_1, getitem_2)

    class submod_0(torch.nn.Module):
        def forward(self, primals_1: "f32[3]", primals_2: "f32[3]"):
            # No stacktrace found for following nodes
            clone_default: "f32[3]" = torch.ops.aten.clone.default(primals_1);  primals_1 = None
            clone_default_1: "f32[3]" = torch.ops.aten.clone.default(primals_2);  primals_2 = None

            # File: /tmp/ipykernel_78130/3551938950.py:41 in compiled_mul_train, code: return mul_vectors_op(x, y).sum()
            mul_vectors_default: "f32[3]" = torch.ops.pallas.mul_vectors.default(clone_default, clone_default_1)
            

Loss:    32.0
x grad:  tensor([4., 5., 6.])
y grad:  tensor([1., 2., 3.])


/home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/torch/_functorch/_aot_autograd/runtime_wrappers.py:2918: UserWarning: Your compiler for AOTAutograd is returning a function that doesn't take boxed arguments. Please wrap it with functorch.compile.make_boxed_func or handle the boxed arguments yourself. See https://github.com/pytorch/pytorch/pull/83137#issuecomment-1211320670 for rationale.
  out = call_func_at_runtime_with_args(


## 5. Verification

A quick sanity check that the Pallas integration is working correctly.

In [6]:
def double_kernel(i_ref, o_ref):
  o_ref[...] = i_ref[...] * 2.0

test_input = torch.tensor([10.0, 20.0, 30.0], device="tpu")

def double_jax_fn(x: jax.Array) -> jax.Array:
  return pl.pallas_call(
      double_kernel, out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype)
  )(x)

double_fn = pallas.jax_op(
    "pallas::double",
    double_jax_fn,
)

test_output = double_fn(test_input)

expected = torch.tensor([20.0, 40.0, 60.0])
torch.testing.assert_close(
    test_output.cpu(), expected
)  # TORCH_ASSERT_CLOSE_OK=<Using standard PyTorch assert for user facing documentation only.>

print(f"Input:    {test_input.cpu()}")
print(f"Output:   {test_output.cpu()}")
print(f"Expected: {expected}")
print("\n✅ Verification complete: Pallas integration functional")

Input:    tensor([10., 20., 30.])
Output:   tensor([20., 40., 60.])
Expected: tensor([20., 40., 60.])

✅ Verification complete: Pallas integration functional


### 🛠️ Summary

| Concept | Key Takeaway |
| :--- | :--- |
| **`pallas.jax_op`** | Decorator to define a TPU kernel as a PyTorch custom operation by wrapping a JAX function |
| **`pl.pallas_call`** | Used to wrap a Pallas kernel function into a JAX function |
| **`_ref[...]` syntax** | Used in these examples to read/write local memory references managed by Pallas |
| **Multiple outputs** | Provide a list of `jax.ShapeDtypeStruct` to `out_shape` in `pl.pallas_call`. |
| **Autograd integration** | Register a backward kernel on the forward `CustomOpDef` returned by `jax_op` using `register_autograd` |
| **`torch.compile`** | Use `backend="tpu"` for full graph fusion with custom kernels |